# Colab · Zero-shot baseline (`whisper-small`, Common Voice 27.0 `tr` test)

Runs `src.evaluation.evaluate_asr` on a Colab GPU. All logic lives in the repository; this
notebook only sets up the runtime.

**Before running**

1. *Runtime → Change runtime type → T4 GPU.*
2. On your computer: `python scripts/pack_data.py --splits test` and upload
   `data/colab/cv-corpus-27.0-2026-09-11-test.tar` to Google Drive (folder in `DRIVE_DIR` below).
3. Set `REPO_URL` below.

Results are copied to `DRIVE_DIR/results/`. Download that folder into the repository's
`results/` and commit it.

In [ ]:
REPO_URL = ""  # e.g. https://github.com/<user>/turkish-asr-summarization.git
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/turkish-asr"  # holds the data .tar; results go here too
DATA_TAR = "cv-corpus-27.0-2026-09-11-test.tar"
REPO_DIR = "/content/turkish-asr-summarization"

assert REPO_URL, "Set REPO_URL first"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Code and dependencies

Colab's preinstalled PyTorch is kept; only the libraries the evaluation needs are installed, at the versions pinned in `requirements.txt`.

In [ ]:
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
PKGS = "transformers|accelerate|jiwer|soundfile|librosa|pyyaml"
!pip install -q $(grep -E '^({PKGS})==' requirements.txt)
!pip install -q --no-deps -e .
!python scripts/check_env.py

## Data

Extract to the VM's local disk (`/content/data`, as in `configs/data_colab.yaml`); reading thousands of clips from Drive directly is slow.

In [ ]:
!mkdir -p /content/data
!tar -xf "{DRIVE_DIR}/{DATA_TAR}" -C /content/data
!ls /content/data/cv-corpus-27.0-2026-09-11/tr
!ls /content/data/cv-corpus-27.0-2026-09-11/tr/clips | wc -l

## Smoke run (64 clips)

Written to `/content/smoke`, not to `results/`.

In [ ]:
EVAL = "-m src.evaluation.evaluate_asr --config configs/eval_baseline.yaml"
COLAB_DATA = "--data-config configs/data_colab.yaml"
!python {EVAL} {COLAB_DATA} --max-samples 64 --output-dir /content/smoke

## Full test split

In [ ]:
!python {EVAL} {COLAB_DATA}

## Save results to Drive

In [ ]:
import json
from pathlib import Path

out = Path("results/baseline/whisper-small-zeroshot")
report = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
assert report["is_full_split"], "metrics.json is not from the full test split"
print(json.dumps({k: report[k] for k in ["model", "num_clips", "metrics", "runtime"]}, indent=2))

!mkdir -p "{DRIVE_DIR}/results/baseline"
!cp -r {out} "{DRIVE_DIR}/results/baseline/"
!ls -la "{DRIVE_DIR}/results/baseline/whisper-small-zeroshot"